# EDA

### Goal of this notebook:
1. Audit the merged NHANES adults dataset
2. Identify candidate target variables for fatigue / low energy / sleep / depression proxies
3. Build a clinically plausible core feature set
4. Assess missingness, subgroup coverage, and feature usefulness
5. Produce a recommendation for next modeling / MVP steps

## Import & Settings

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

In [7]:
DATA_PATH = Path("../data/processed/nhanes_merged_adults.csv")

df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape}")
display(df.head())

Shape: (7437, 371)


C:\Users\Philipp\AppData\Local\Temp\ipykernel_38404\357236.py:3: DtypeWarning: Columns (0: medication_18, 1: medication_19, 2: medication_20, 3: medication_21, 4: medication_22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH)


,SEQN,RIAGENDR,RIDAGEYR,RIDRETH3,RIDRETH1,RIDEXPRG,DMDEDUC2,DMDMARTZ,DMDBORN4,INDFMPIR,WTMECPRP,WTINTPRP,SDMVPSU,SDMVSTRA,gender,ethnicity,education,marital_status,pregnancy_status,country_of_birth,calories,protein,carbs,fat,iron,vitamin_b12,vitamin_d,folate,magnesium,zinc,height_cm,weight_kg,bmi,waist_cm,hip_cm,sbp_1,sbp_2,sbp_3,dbp_1,dbp_2,dbp_3,pulse_1,pulse_2,pulse_3,liver_cap_dbm,liver_stiffness_kpa,liver_exam_status,liver_valid_measures,liver_stiffness_iqr_ratio,oral_gum_problem_yesno,oral_decayed_teeth_yesno,oral_hygiene_issue_yesno,any_implant,any_root_caries,any_root_restoration_caries,fasting_hours_part,fasting_minutes_part,fasting_glucose_mg_dl,insulin_uU_ml,total_cholesterol_mg_dl,hdl_cholesterol_mg_dl,triglycerides_mg_dl,ferritin_ng_ml,serum_iron_ug_dl,tibc_ug_dl,transferrin_saturation_pct,transferrin_receptor_mg_l,uacr_mg_g,serum_creatinine_mg_dl,bun_mg_dl,alt_u_l,ast_u_l,ggt_u_l,alp_u_l,total_bilirubin_mg_dl,serum_albumin_g_dl,alq111___ever_had_a_drink_of_any_kind_of_alcohol,alq121___past_12_mo_how_often_drink_alcoholic_bev,alq130___avg_#_alcoholic_drinks/day___past_12_mos,alq142___#_days_have_4_or_5_drinks/past_12_mos,alq270___#_times_4_5_drinks_in_2hrs/past_12_mos,alq280___#_times_8+_drinks_in_1_day/past_12_mos,alq290___#_times_12+_drinks_in_1_day/past_12_mos,alq151___ever_have_4/5_or_more_drinks_every_day?,alq170___past_30_days_#_times_4_5_drinks_on_an_oc,bpq020___ever_told_you_had_high_blood_pressure,bpq030___told_had_high_blood_pressure___2+_times,bpd035___age_told_had_hypertension,bpq040a___taking_prescription_for_hypertension,bpq050a___now_taking_prescribed_medicine_for_hbp,bpq080___doctor_told_you___high_cholesterol_level,bpq060___ever_had_blood_cholesterol_checked,bpq070___when_blood_cholesterol_last_checked,bpq090d___told_to_take_prescriptn_for_cholesterol,bpq100d___now_taking_prescribed_medicine,cdq001___sp_ever_had_pain_or_discomfort_in_chest,cdq002___sp_get_it_walking_uphill_or_in_a_hurry,cdq003___during_an_ordinary_pace_on_level_ground,cdq004___if_so_does_sp_continue_or_slow_down,cdq005___does_standing_relieve_pain/discomfort,...,rxqseen___medicine_container_seen_by_interviewer,rxddays___number_of_days_taken_medicine,rxdrsc1___icd_10_cm_code_1,rxdrsc2___icd_10_cm_code_2,rxdrsc3___icd_10_cm_code_3,rxdrsd1___icd_10_cm_code_1_description,rxdrsd2___icd_10_cm_code_2_description,rxdrsd3___icd_10_cm_code_3_description,rxdcount___number_of_prescription_medicines_taken,slq300___usual_sleep_time_on_weekdays_or_workdays,slq310___usual_wake_time_on_weekdays_or_workdays,sld012___sleep_hours___weekdays_or_workdays,slq320___usual_sleep_time_on_weekends,slq330___usual_wake_time_on_weekends,sld013___sleep_hours___weekends,slq030___how_often_do_you_snore?,slq040___how_often_do_you_snort_or_stop_breathing,slq050___ever_told_doctor_had_trouble_sleeping?,smq020___smoked_at_least_100_cigarettes_in_life,smd030___age_started_smoking_cigarettes_regularly,smq040___do_you_now_smoke_cigarettes?,smq050q___how_long_since_quit_smoking_cigarettes,smq050u___unit_of_measure_(day/week/month/year),smd057___#_cigarettes_smoked_per_day_when_quit,smq078___how_soon_after_waking_do_you_smoke,smd641___#_days_smoked_cigs_during_past_30_days,smd650___avg_#_cigarettes/day_during_past_30_days,smd100fl___cigarette_filter_type,smd100mn___cigarette_menthol_indicator,smq670___tried_to_quit_smoking,smq621___cigarettes_smoked_in_entire_life,smd630___age_first_smoked_whole_cigarette,smaquex2___questionnaire_mode_flag,whd010___current_self_reported_height_(inches),whd020___current_self_reported_weight_(pounds),whq030___how_do_you_consider_your_weight,"whq040___like_to_weigh_more,_less_or_same",whd050___self_reported_weight___1_yr_ago_(pounds),whq060___weight_change_intentional,whq070___tried_to_lose_weight_in_past_year,whd080a___ate_less_to_lose_weight,whd080b___switched_to_foods_with_lower_calories,whd080c___ate_less_fat_to_lose_weight,whd080d___exercised_to_lose_weight,whd080e___skipped_meals,whd080f___ate_diet_foods_or_products,whd080g___used_a_liq

In [8]:
def dataset_overview(dataframe):
    overview = pd.DataFrame({
        "column": dataframe.columns,
        "dtype": dataframe.dtypes.astype(str).values,
        "missing_n": dataframe.isna().sum().values,
        "missing_pct": (dataframe.isna().mean().values * 100).round(2),
        "n_unique": dataframe.nunique(dropna=True).values
    }).sort_values("missing_pct", ascending=False)
    return overview

overview = dataset_overview(df)
display(overview.head(30))

print("Rows:", len(df))
print("Columns:", df.shape[1])
print("Numeric columns:", df.select_dtypes(include=np.number).shape[1])
print("Non-numeric columns:", df.select_dtypes(exclude=np.number).shape[1])

,column,dtype,missing_n,missing_pct,n_unique
301,smq621___cigarettes_smoked_in_entire_life,float64,7437,100.00,0
302,smd630___age_first_smoked_whole_cigarette,float64,7437,100.00,0
174,mcq151___age_in_years_at_first_menstrual_period,float64,7437,100.00,0
173,mcq149___menstrual_periods_started_yet?,float64,7437,100.00,0
365,medication_22,str,7435,99.97,2
364,medication_21,str,7435,99.97,2
363,medication_20,str,7434,99.96,3
210,mcq230c___what_kind_of_cancer_third_mention,float64,7432,99.93,4
362,medication_19,str,7432,99.93,5
196,mcq510b___liver_condition_non_alcoholic_fatty_liver,float64,7431,99.92,1


Rows: 7437
Columns: 371
Numeric columns: 330
Non-numeric columns: 41


## Rename column names (domain demographics) and remove doubles

In [21]:
from pathlib import Path
import pandas as pd

# Sicherheitskopie
df_clean = df.copy()

# 1) Kryptische NHANES-Spalten droppen, wenn bereits eine lesbare Version existiert
drop_if_readable_exists = {
    "RIAGENDR": "gender",
    "RIDRETH1": "ethnicity",
    "RIDRETH3": "ethnicity",
    "DMDEDUC2": "education",
    "DMDMARTZ": "marital_status",
    "RIDEXPRG": "pregnancy_status",
    "DMDBORN4": "country_of_birth",
}

cols_to_drop = [
    raw_col
    for raw_col, readable_col in drop_if_readable_exists.items()
    if raw_col in df_clean.columns and readable_col in df_clean.columns
]

# 2) Spalten umbenennen, für die es noch keine lesbare Version gibt
rename_map = {
    "RIDAGEYR": "age_years",
    "INDFMPIR": "income_poverty_ratio",
    "WTMECPRP": "mec_exam_weight",
    "WTINTPRP": "interview_weight",
    "SDMVPSU": "survey_psu",
    "SDMVSTRA": "survey_stratum",
}

# Nur umbenennen, wenn die Originalspalte existiert und der Zielname noch nicht existiert
rename_map_final = {
    old: new
    for old, new in rename_map.items()
    if old in df_clean.columns and new not in df_clean.columns
}

# Anwenden
df_clean = df_clean.drop(columns=cols_to_drop).rename(columns=rename_map_final)

print("Gedroppt:")
print(cols_to_drop)

print("\nUmbenannt:")
print(rename_map_final)

print("\nNeue Shape:", df_clean.shape)
display(df_clean.head())

Gedroppt:
['RIAGENDR', 'RIDRETH1', 'RIDRETH3', 'DMDEDUC2', 'DMDMARTZ', 'RIDEXPRG', 'DMDBORN4']

Umbenannt:
{'RIDAGEYR': 'age_years', 'INDFMPIR': 'income_poverty_ratio', 'WTMECPRP': 'mec_exam_weight', 'WTINTPRP': 'interview_weight', 'SDMVPSU': 'survey_psu', 'SDMVSTRA': 'survey_stratum'}

Neue Shape: (7437, 364)


,SEQN,age_years,income_poverty_ratio,mec_exam_weight,interview_weight,survey_psu,survey_stratum,gender,ethnicity,education,marital_status,pregnancy_status,country_of_birth,calories,protein,carbs,fat,iron,vitamin_b12,vitamin_d,folate,magnesium,zinc,height_cm,weight_kg,bmi,waist_cm,hip_cm,sbp_1,sbp_2,sbp_3,dbp_1,dbp_2,dbp_3,pulse_1,pulse_2,pulse_3,liver_cap_dbm,liver_stiffness_kpa,liver_exam_status,liver_valid_measures,liver_stiffness_iqr_ratio,oral_gum_problem_yesno,oral_decayed_teeth_yesno,oral_hygiene_issue_yesno,any_implant,any_root_caries,any_root_restoration_caries,fasting_hours_part,fasting_minutes_part,fasting_glucose_mg_dl,insulin_uU_ml,total_cholesterol_mg_dl,hdl_cholesterol_mg_dl,triglycerides_mg_dl,ferritin_ng_ml,serum_iron_ug_dl,tibc_ug_dl,transferrin_saturation_pct,transferrin_receptor_mg_l,uacr_mg_g,serum_creatinine_mg_dl,bun_mg_dl,alt_u_l,ast_u_l,ggt_u_l,alp_u_l,total_bilirubin_mg_dl,serum_albumin_g_dl,alq111___ever_had_a_drink_of_any_kind_of_alcohol,alq121___past_12_mo_how_often_drink_alcoholic_bev,alq130___avg_#_alcoholic_drinks/day___past_12_mos,alq142___#_days_have_4_or_5_drinks/past_12_mos,alq270___#_times_4_5_drinks_in_2hrs/past_12_mos,alq280___#_times_8+_drinks_in_1_day/past_12_mos,alq290___#_times_12+_drinks_in_1_day/past_12_mos,alq151___ever_have_4/5_or_more_drinks_every_day?,alq170___past_30_days_#_times_4_5_drinks_on_an_oc,bpq020___ever_told_you_had_high_blood_pressure,bpq030___told_had_high_blood_pressure___2+_times,bpd035___age_told_had_hypertension,bpq040a___taking_prescription_for_hypertension,bpq050a___now_taking_prescribed_medicine_for_hbp,bpq080___doctor_told_you___high_cholesterol_level,bpq060___ever_had_blood_cholesterol_checked,bpq070___when_blood_cholesterol_last_checked,bpq090d___told_to_take_prescriptn_for_cholesterol,bpq100d___now_taking_prescribed_medicine,cdq001___sp_ever_had_pain_or_discomfort_in_chest,cdq002___sp_get_it_walking_uphill_or_in_a_hurry,cdq003___during_an_ordinary_pace_on_level_ground,cdq004___if_so_does_sp_continue_or_slow_down,cdq005___does_standing_relieve_pain/discomfort,cdq006___how_soon_is_the_pain_relieved,cdq009a___pain_in_right_arm,cdq009b___pain_in_right_chest,cdq009c___pain_in_neck,cdq009d___pain_in_upper_sternum,cdq009e___pain_in_lower_sternum,cdq009f___pain_in_left_chest,...,rxqseen___medicine_container_seen_by_interviewer,rxddays___number_of_days_taken_medicine,rxdrsc1___icd_10_cm_code_1,rxdrsc2___icd_10_cm_code_2,rxdrsc3___icd_10_cm_code_3,rxdrsd1___icd_10_cm_code_1_description,rxdrsd2___icd_10_cm_code_2_description,rxdrsd3___icd_10_cm_code_3_description,rxdcount___number_of_prescription_medicines_taken,slq300___usual_sleep_time_on_weekdays_or_workdays,slq310___usual_wake_time_on_weekdays_or_workdays,sld012___sleep_hours___weekdays_or_workdays,slq320___usual_sleep_time_on_weekends,slq330___usual_wake_time_on_weekends,sld013___sleep_hours___weekends,slq030___how_often_do_you_snore?,slq040___how_often_do_you_snort_or_stop_breathing,slq050___ever_told_doctor_had_trouble_sleeping?,smq020___smoked_at_least_100_cigarettes_in_life,smd030___age_started_smoking_cigarettes_regularly,smq040___do_you_now_smoke_cigarettes?,smq050q___how_long_since_quit_smoking_cigarettes,smq050u___unit_of_measure_(day/week/month/year),smd057___#_cigarettes_smoked_per_day_when_quit,smq078___how_soon_after_waking_do_you_smoke,smd641___#_days_smoked_cigs_during_past_30_days,smd650___avg_#_cigarettes/day_during_past_30_days,smd100fl___cigarette_filter_type,smd100mn___cigarette_menthol_indicator,smq670___tried_to_quit_smoking,smq621___cigarettes_smoked_in_entire_life,smd630___age_first_smoked_whole_cigarette,smaquex2___questionnaire_mode_flag,whd010___current_self_reported_height_(inches),whd020___current_self_reported_weight_(pounds),whq030___how_do_you_consider_your_weight,"whq040___like_to_weigh_more,_less_or_same",whd050___self_reported_weight___1_yr_ago_(pounds),whq060___weight_change_intentional,whq070___tried_to_lose_weight_in_past_year,whd080a___ate_less_to_lose_weight,whd080b___switc

In [23]:
output_path = Path("../data/processed/nhanes_merged_adults_cleaned.csv")
df_clean.to_csv(output_path, index=False)
print(f"Gespeichert unter: {output_path}")

Gespeichert unter: ..\data\processed\nhanes_merged_adults_cleaned.csv


## Domain Mapping

In [26]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

BASE_PATH = Path("../data/processed")

SOURCE_FILES = {
    "demographics": BASE_PATH / "demo_all_adults.csv",
    "dietary": BASE_PATH / "dietary_clean.csv",
    "examination": BASE_PATH / "examination_clean.csv",
    "laboratory": BASE_PATH / "laboratory_clean.csv",
    "questionnaire": BASE_PATH / "merged_questionnaire.csv",
}

In [27]:
def normalize_colname(col):
    col = str(col).strip().lower()
    col = re.sub(r"[^a-z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col).strip("_")
    return col

In [28]:
catalog_rows = []

for domain, path in SOURCE_FILES.items():
    cols = pd.read_csv(path, nrows=0).columns.tolist()
    
    for col in cols:
        catalog_rows.append({
            "source_domain": domain,
            "source_file": path.name,
            "source_column": col,
            "source_column_norm": normalize_colname(col)
        })

source_catalog = pd.DataFrame(catalog_rows)

display(source_catalog.head())
print("Anzahl Katalog-Zeilen:", len(source_catalog))

,source_domain,source_file,source_column,source_column_norm
0,demographics,demo_all_adults.csv,SEQN,seqn
1,demographics,demo_all_adults.csv,RIAGENDR,riagendr
2,demographics,demo_all_adults.csv,RIDAGEYR,ridageyr
3,demographics,demo_all_adults.csv,RIDRETH3,ridreth3
4,demographics,demo_all_adults.csv,RIDRETH1,ridreth1


Anzahl Katalog-Zeilen: 375


In [29]:
duplicate_source_cols = (
    source_catalog.groupby("source_column_norm")
    .agg(
        n_sources=("source_file", "nunique"),
        source_files=("source_file", lambda x: " | ".join(sorted(set(x)))),
        domains=("source_domain", lambda x: " | ".join(sorted(set(x))))
    )
    .reset_index()
)

duplicates_only = duplicate_source_cols[duplicate_source_cols["n_sources"] > 1]
display(duplicates_only.head(20))

,source_column_norm,n_sources,source_files,domains
294,seqn,5,demo_all_adults.csv | dietary_clean.csv | examination_clean.csv | laboratory_clean.csv | merged_questionnaire.csv,demographics | dietary | examination | laboratory | questionnaire


In [30]:
def dataset_overview(dataframe):
    overview = pd.DataFrame({
        "column": dataframe.columns,
        "dtype": dataframe.dtypes.astype(str).values,
        "missing_n": dataframe.isna().sum().values,
        "missing_pct": (dataframe.isna().mean().values * 100).round(2),
        "n_unique": dataframe.nunique(dropna=True).values
    })
    return overview.sort_values("missing_pct", ascending=False)

overview_clean = dataset_overview(df_clean)
display(overview_clean.head(30))

,column,dtype,missing_n,missing_pct,n_unique
167,mcq151___age_in_years_at_first_menstrual_period,float64,7437,100.00,0
295,smd630___age_first_smoked_whole_cigarette,float64,7437,100.00,0
294,smq621___cigarettes_smoked_in_entire_life,float64,7437,100.00,0
166,mcq149___menstrual_periods_started_yet?,float64,7437,100.00,0
358,medication_22,str,7435,99.97,2
357,medication_21,str,7435,99.97,2
356,medication_20,str,7434,99.96,3
355,medication_19,str,7432,99.93,5
203,mcq230c___what_kind_of_cancer_third_mention,float64,7432,99.93,4
189,mcq510b___liver_condition_non_alcoholic_fatty_liver,float64,7431,99.92,1


In [31]:
overview["column_norm"] = overview["column"].apply(normalize_colname)

source_lookup = (
    source_catalog.groupby("source_column_norm")
    .agg(
        source_domain=("source_domain", lambda x: " | ".join(sorted(set(x)))),
        source_file=("source_file", lambda x: " | ".join(sorted(set(x)))),
        source_column=("source_column", lambda x: " | ".join(sorted(set(x))))
    )
    .reset_index()
)

overview = overview.merge(
    source_lookup,
    left_on="column_norm",
    right_on="source_column_norm",
    how="left"
)

overview["domain"] = overview["source_domain"]
overview["mapping_method"] = np.where(
    overview["domain"].notna(),
    "exact_header_match",
    pd.NA
)

display(overview.head(20))

,column,dtype,missing_n,missing_pct,n_unique,column_norm,source_column_norm_x,source_domain_x,source_file_x,source_column_x,domain,mapping_method,missing_bucket,source_column_norm_y,source_domain_y,source_file_y,source_column_y,source_column_norm,source_domain,source_file,source_column
0,SEQN,int64,0,0.00,7437,seqn,seqn,demographics | dietary | examination | laboratory | questionnaire,demo_all_adults.csv | dietary_clean.csv | examination_clean.csv | laboratory_clean.csv | merged_questionnaire.csv,SEQN,demographics | dietary | examination | laboratory | questionnaire,exact_header_match,0-5%,seqn,demographics | dietary | examination | laboratory | questionnaire,demo_all_adults.csv | dietary_clean.csv | examination_clean.csv | laboratory_clean.csv | merged_questionnaire.csv,SEQN,seqn,demographics | dietary | examination | laboratory | questionnaire,demo_all_adults.csv | dietary_clean.csv | examination_clean.csv | laboratory_clean.csv | merged_questionnaire.csv,SEQN
1,RIAGENDR,float64,0,0.00,2,riagendr,riagendr,demographics,demo_all_adults.csv,RIAGENDR,demographics,exact_header_match,0-5%,riagendr,demographics,demo_all_adults.csv,RIAGENDR,riagendr,demographics,demo_all_adults.csv,RIAGENDR
2,RIDAGEYR,float64,0,0.00,48,ridageyr,ridageyr,demographics,demo_all_adults.csv,RIDAGEYR,demographics,exact_header_match,0-5%,ridageyr,demographics,demo_all_adults.csv,RIDAGEYR,ridageyr,demographics,demo_all_adults.csv,RIDAGEYR
3,RIDRETH3,float64,0,0.00,6,ridreth3,ridreth3,demographics,demo_all_adults.csv,RIDRETH3,demographics,exact_header_match,0-5%,ridreth3,demographics,demo_all_adults.csv,RIDRETH3,ridreth3,demographics,demo_all_adults.csv,RIDRETH3
4,RIDRETH1,float64,0,0.00,5,ridreth1,ridreth1,demographics,demo_all_adults.csv,RIDRETH1,demographics,exact_header_match,0-5%,ridreth1,demographics,demo_all_adults.csv,RIDRETH1,ridreth1,demographics,demo_all_adults.csv,RIDRETH1
5,RIDEXPRG,float64,5563,74.80,3,ridexprg,ridexprg,demographics,demo_all_adults.csv,RIDEXPRG,demographics,exact_header_match,50-80%,ridexprg,demographics,demo_all_adults.csv,RIDEXPRG,ridexprg,demographics,demo_all_adults.csv,RIDEXPRG
6,DMDEDUC2,float64,461,6.20,7,dmdeduc2,dmdeduc2,demographics,demo_all_adults.csv,DMDEDUC2,demographics,exact_header_match,5-20%,dmdeduc2,demographics,demo_all_adults.csv,DMDEDUC2,dmdeduc2,demographics,demo_all_adults.csv,DMDEDUC2
7,DMDMARTZ,float64,461,6.20,4,dmdmartz,dmdmartz,demographics,demo_all_adults.csv,DMDMARTZ,demographics,exact_header_match,5-20%,dmdmartz,demographics,demo_all_adults.csv,DMDMARTZ,dmdmartz,demographics,demo_all_adults.csv,DMDMARTZ
8,DMDBORN4,float64,0,0.00,4,dmdborn4,dmdborn4,demographics,demo_all_adults.csv,DMDBORN4,demographics,exact_header_match,0-5%,dmdborn4,demographics,demo_all_adults.csv,DMDBORN4,dmdborn4,demographics,demo_all_adults.csv,DMDBORN4
9,INDFMPIR,float64,1118,15.03,460,indfmpir,indfmpir,demographics,demo_all_adults.csv,INDFMPIR,demographics,exact_header_match,5-20%,indfmpir,demographics,demo_all_adults.csv,INDFMPIR,indfmpir,demographics,demo_all_adults.csv,INDFMPIR


In [32]:
overview.to_csv("03_overview_with_domain_mapping.csv", index=False)
print("Saved: 03_overview_with_domain_mapping.csv")

Saved: 03_overview_with_domain_mapping.csv


### doubles: 

In [33]:
demographic_cols = overview[
    overview["domain"].str.contains("demographics", na=False)
]

display(demographic_cols)

,column,dtype,missing_n,missing_pct,n_unique,column_norm,source_column_norm_x,source_domain_x,source_file_x,source_column_x,domain,mapping_method,missing_bucket,source_column_norm_y,source_domain_y,source_file_y,source_column_y,source_column_norm,source_domain,source_file,source_column
0,SEQN,int64,0,0.00,7437,seqn,seqn,demographics | dietary | examination | laboratory | questionnaire,demo_all_adults.csv | dietary_clean.csv | examination_clean.csv | laboratory_clean.csv | merged_questionnaire.csv,SEQN,demographics | dietary | examination | laboratory | questionnaire,exact_header_match,0-5%,seqn,demographics | dietary | examination | laboratory | questionnaire,demo_all_adults.csv | dietary_clean.csv | examination_clean.csv | laboratory_clean.csv | merged_questionnaire.csv,SEQN,seqn,demographics | dietary | examination | laboratory | questionnaire,demo_all_adults.csv | dietary_clean.csv | examination_clean.csv | laboratory_clean.csv | merged_questionnaire.csv,SEQN
1,RIAGENDR,float64,0,0.00,2,riagendr,riagendr,demographics,demo_all_adults.csv,RIAGENDR,demographics,exact_header_match,0-5%,riagendr,demographics,demo_all_adults.csv,RIAGENDR,riagendr,demographics,demo_all_adults.csv,RIAGENDR
2,RIDAGEYR,float64,0,0.00,48,ridageyr,ridageyr,demographics,demo_all_adults.csv,RIDAGEYR,demographics,exact_header_match,0-5%,ridageyr,demographics,demo_all_adults.csv,RIDAGEYR,ridageyr,demographics,demo_all_adults.csv,RIDAGEYR
3,RIDRETH3,float64,0,0.00,6,ridreth3,ridreth3,demographics,demo_all_adults.csv,RIDRETH3,demographics,exact_header_match,0-5%,ridreth3,demographics,demo_all_adults.csv,RIDRETH3,ridreth3,demographics,demo_all_adults.csv,RIDRETH3
4,RIDRETH1,float64,0,0.00,5,ridreth1,ridreth1,demographics,demo_all_adults.csv,RIDRETH1,demographics,exact_header_match,0-5%,ridreth1,demographics,demo_all_adults.csv,RIDRETH1,ridreth1,demographics,demo_all_adults.csv,RIDRETH1
5,RIDEXPRG,float64,5563,74.80,3,ridexprg,ridexprg,demographics,demo_all_adults.csv,RIDEXPRG,demographics,exact_header_match,50-80%,ridexprg,demographics,demo_all_adults.csv,RIDEXPRG,ridexprg,demographics,demo_all_adults.csv,RIDEXPRG
6,DMDEDUC2,float64,461,6.20,7,dmdeduc2,dmdeduc2,demographics,demo_all_adults.csv,DMDEDUC2,demographics,exact_header_match,5-20%,dmdeduc2,demographics,demo_all_adults.csv,DMDEDUC2,dmdeduc2,demographics,demo_all_adults.csv,DMDEDUC2
7,DMDMARTZ,float64,461,6.20,4,dmdmartz,dmdmartz,demographics,demo_all_adults.csv,DMDMARTZ,demographics,exact_header_match,5-20%,dmdmartz,demographics,demo_all_adults.csv,DMDMARTZ,dmdmartz,demographics,demo_all_adults.csv,DMDMARTZ
8,DMDBORN4,float64,0,0.00,4,dmdborn4,dmdborn4,demographics,demo_all_adults.csv,DMDBORN4,demographics,exact_header_match,0-5%,dmdborn4,demographics,demo_all_adults.csv,DMDBORN4,dmdborn4,demographics,demo_all_adults.csv,DMDBORN4
9,INDFMPIR,float64,1118,15.03,460,indfmpir,indfmpir,demographics,demo_all_adults.csv,INDFMPIR,demographics,exact_header_match,5-20%,indfmpir,demographics,demo_all_adults.csv,INDFMPIR,indfmpir,demographics,demo_all_adults.csv,INDFMPIR


## Missingness Audit

In [34]:
high_missing = overview.sort_values("missing_pct", ascending=False)
display(high_missing.head(50))

,column,dtype,missing_n,missing_pct,n_unique,column_norm,source_column_norm_x,source_domain_x,source_file_x,source_column_x,domain,mapping_method,missing_bucket,source_column_norm_y,source_domain_y,source_file_y,source_column_y,source_column_norm,source_domain,source_file,source_column
301,smq621___cigarettes_smoked_in_entire_life,float64,7437,100.00,0,smq621_cigarettes_smoked_in_entire_life,smq621_cigarettes_smoked_in_entire_life,questionnaire,merged_questionnaire.csv,smq621___cigarettes_smoked_in_entire_life,questionnaire,exact_header_match,95-100%,smq621_cigarettes_smoked_in_entire_life,questionnaire,merged_questionnaire.csv,smq621___cigarettes_smoked_in_entire_life,smq621_cigarettes_smoked_in_entire_life,questionnaire,merged_questionnaire.csv,smq621___cigarettes_smoked_in_entire_life
302,smd630___age_first_smoked_whole_cigarette,float64,7437,100.00,0,smd630_age_first_smoked_whole_cigarette,smd630_age_first_smoked_whole_cigarette,questionnaire,merged_questionnaire.csv,smd630___age_first_smoked_whole_cigarette,questionnaire,exact_header_match,95-100%,smd630_age_first_smoked_whole_cigarette,questionnaire,merged_questionnaire.csv,smd630___age_first_smoked_whole_cigarette,smd630_age_first_smoked_whole_cigarette,questionnaire,merged_questionnaire.csv,smd630___age_first_smoked_whole_cigarette
174,mcq151___age_in_years_at_first_menstrual_period,float64,7437,100.00,0,mcq151_age_in_years_at_first_menstrual_period,mcq151_age_in_years_at_first_menstrual_period,questionnaire,merged_questionnaire.csv,mcq151___age_in_years_at_first_menstrual_period,questionnaire,exact_header_match,95-100%,mcq151_age_in_years_at_first_menstrual_period,questionnaire,merged_questionnaire.csv,mcq151___age_in_years_at_first_menstrual_period,mcq151_age_in_years_at_first_menstrual_period,questionnaire,merged_questionnaire.csv,mcq151___age_in_years_at_first_menstrual_period
173,mcq149___menstrual_periods_started_yet?,float64,7437,100.00,0,mcq149_menstrual_periods_started_yet,mcq149_menstrual_periods_started_yet,questionnaire,merged_questionnaire.csv,mcq149___menstrual_periods_started_yet?,questionnaire,exact_header_match,95-100%,mcq149_menstrual_periods_started_yet,questionnaire,merged_questionnaire.csv,mcq149___menstrual_periods_started_yet?,mcq149_menstrual_periods_started_yet,questionnaire,merged_questionnaire.csv,mcq149___menstrual_periods_started_yet?
365,medication_22,str,7435,99.97,2,medication_22,medication_22,questionnaire,merged_questionnaire.csv,medication_22,questionnaire,exact_header_match,95-100%,medication_22,questionnaire,merged_questionnaire.csv,medication_22,medication_22,questionnaire,merged_questionnaire.csv,medication_22
364,medication_21,str,7435,99.97,2,medication_21,medication_21,questionnaire,merged_questionnaire.csv,medication_21,questionnaire,exact_header_match,95-100%,medication_21,questionnaire,merged_questionnaire.csv,medication_21,medication_21,questionnaire,merged_questionnaire.csv,medication_21
363,medication_20,str,7434,99.96,3,medication_20,medication_20,questionnaire,merged_questionnaire.csv,medication_20,questionnaire,exact_header_match,95-100%,medication_20,questionnaire,merged_questionnaire.csv,medication_20,medication_20,questionnaire,merged_questionnaire.csv,medication_20
210,mcq230c___what_kind_of_cancer_third_mention,float64,7432,99.93,4,mcq230c_what_kind_of_cancer_third_mention,mcq230c_what_kind_of_cancer_third_mention,questionnaire,merged_questionnaire.csv,mcq230c___what_kind_of_cancer_third_mention,questionnaire,exact_header_match,95-100%,mcq230c_what_kind_of_cancer_third_mention,questionnaire,merged_questionnaire.csv,mcq230c___what_kind_of_cancer_third_mention,mcq230c_what_kind_of_cancer_third_mention,questionnaire,merged_questionnaire.csv,mcq230c___what_kind_of_cancer_third_mention
362,medication_19,str,7432,99.93,5,medication_19,medication_19,questionnaire,merged_questionnaire.csv,medication_19,questionnaire,exact_header_match,95-100%,medication_19,questionnaire,merged_questionnaire.csv,medication_19,medication_19

In [35]:
bins = [0, 5, 20, 50, 80, 95, 100]
labels = ["0-5%", "5-20%", "20-50%", "50-80%", "80-95%", "95-100%"]

overview["missing_bucket"] = pd.cut(
    overview["missing_pct"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

display(overview["missing_bucket"].value_counts().sort_index())

missing_bucket
0-5%       69
5-20%      79
20-50%     17
50-80%     70
80-95%     64
95-100%    72
Name: count, dtype: int64